# Learn: valid solid vs self-intersecting geometry

**Why this notebook exists:** in the Phase-5 re-run, 3 of 7 designs failed with errors like
`Invalid boundary mesh (overlapping facets)`. This notebook shows *visually* what that means —
a 3D shape that passes through itself and therefore can't be meshed or 3D-printed. Rotate the
3D plots to see it. Runs anywhere (just `numpy` + `matplotlib`).

## 1. The idea

A 3D-printable part must be a **clean solid**: one surface that seals a volume without ever
crossing itself — like a balloon. If a shape curls or bulges until one wall pokes through
another, the computer can't tell *inside* from *outside* at the crossing, so the mesher (and a
slicer) reject it. A fan blade is basically a **curved (cambered) panel**; crank the camber far
enough and the panel curls back *through itself*. That's exactly what happened to the extreme
high-airflow designs the optimizer chased.

## 2. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 3. Model a blade as a curled panel

We take a flat panel of fixed arc-length (`chord`) and **curl it through an angle** (`curl_deg`
= how much camber). Small angle = gentle curve (a real, valid blade). Past ~360° the panel
wraps all the way around and starts overlapping itself. We then sweep that cross-section along
the blade's length (`span`) to get the 3D surface.

In [ ]:
def blade(curl_deg, chord=1.0, span=1.6, n=90, m=24):
    curl = np.radians(curl_deg)
    R = chord / curl                                  # tighter roll -> more camber
    s = np.linspace(0, curl, n)                       # angle traversed along the chord
    x_cs, z_cs = R * np.sin(s), R * (1 - np.cos(s))   # the cross-section curve
    y = np.linspace(0, span, m)                       # sweep along the blade length
    X = np.tile(x_cs, (m, 1)); Z = np.tile(z_cs, (m, 1)); Y = np.tile(y[:, None], (1, n))
    return x_cs, z_cs, X, Y, Z

## 4. Plot helper

Left = the 2D cross-section (the curve the blade is swept from). Right = the 3D surface.
Watch the cross-section: a valid blade is an open **C**; an invalid one **loops back through
itself**.

In [ ]:
def show(curl_deg, title, color):
    x_cs, z_cs, X, Y, Z = blade(curl_deg)
    fig = plt.figure(figsize=(11, 4))
    ax = fig.add_subplot(1, 2, 1)
    ax.plot(x_cs, z_cs, color=color, lw=2.5)
    ax.set_aspect("equal"); ax.grid(alpha=0.3)
    ax.set_title(f"cross-section (2D)\n{title}"); ax.set_xlabel("x"); ax.set_ylabel("z")
    ax3 = fig.add_subplot(1, 2, 2, projection="3d")
    ax3.plot_surface(X, Y, Z, color=color, alpha=0.55, edgecolor="k", linewidth=0.15)
    ax3.set_title(f"3D blade surface\n{title}"); ax3.view_init(elev=22, azim=-60)
    ax3.set_xlabel("x"); ax3.set_ylabel("span"); ax3.set_zlabel("z")
    plt.tight_layout(); plt.show()

## 5. A VALID blade (gentle camber) — clean sheet, meshes fine

In [ ]:
show(70, "VALID (camber ~70 deg)", "seagreen")

## 6. A SELF-INTERSECTING blade (extreme camber) — the surface passes through itself

This is what an extreme high-`J_fan` design looks like in 3D. The cross-section spirals past
itself; the 3D surface overlaps. **Rotate it** (drag) and you'll see the sheet crossing through
its own body — that's the `overlapping facets` the mesher refused, and it's un-printable.

In [ ]:
show(430, "SELF-INTERSECTING (camber ~430 deg)", "crimson")

## 7. Takeaway

- The **2D slice** the optimizer uses is a flat cross-section — it *scores airflow* but never
  builds the full 3D solid, so it **cannot see** this self-intersection. It happily gave these
  designs high `J_fan` and put them on the Pareto front.
- The **3D verification** (Phase 5) *does* build the solid — which is why it caught them (as
  mesh failures).
- The fix (queued for V1.5, `docs/V2_backlog.md`): build the 3D CAD solid inside the optimizer
  and **reject self-intersecting geometry**, so it never chases un-buildable shapes.